# 평가 결과 분석 예시 (CIFAR-100 CNN PoC)

web에서 실행한 평가 결과(`storage/models/*/evaluations/*`)를 읽어 보고서용 통계 분석을 합니다.

- 커널: `cnn-poc-trainer` (등록: `conda run -n cnn-poc-trainer python -m ipykernel install --user --name cnn-poc-trainer`)
- 먼저 web의 **평가** 화면에서 모델을 평가해 두어야 합니다.

In [ ]:
import json, sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from app.core.stats import wilson_ci, mcnemar

MODELS = ROOT / 'storage' / 'models'
evals = []
for meta_path in sorted(MODELS.glob('*/meta.json')):
    meta = json.loads(meta_path.read_text())
    for m in sorted(meta_path.parent.glob('evaluations/*/metrics.json')):
        evals.append((meta, json.loads(m.read_text()), m.parent))
print(f'평가 결과 {len(evals)}개')

## 1. 모델별 test 정확도와 95% 신뢰구간 (Wilson)

In [ ]:
for meta, m, _ in evals:
    lo, hi = m['top1_ci']
    print(f"{meta['name'][:40]:40s} width={meta['hparams']['width']:<5} top-1={m['top1']:.4f} [{lo:.4f}, {hi:.4f}]  top-5={m['top5']:.4f}  ECE={m['ece']:.4f}")

## 2. Bootstrap 신뢰구간과 비교

표본(test 이미지)을 복원추출해 정확도 분포를 만들고, Wilson 구간과 비교합니다.

In [ ]:
rng = np.random.default_rng(0)
for meta, m, d in evals:
    z = np.load(d / 'preds.npz')
    correct = (z['y_true'] == z['y_pred']).astype(float)
    boots = [correct[rng.integers(0, len(correct), len(correct))].mean() for _ in range(2000)]
    lo, hi = np.percentile(boots, [2.5, 97.5])
    print(f"{meta['name'][:40]:40s} bootstrap [{lo:.4f}, {hi:.4f}]  vs Wilson [{m['top1_ci'][0]:.4f}, {m['top1_ci'][1]:.4f}]")

## 3. 클래스별 정확도 분포

같은 모델이라도 클래스마다 정확도 차이가 큽니다. 어떤 클래스가 어려운지 확인합니다.

In [ ]:
if evals:
    meta, m, _ = evals[-1]
    acc = np.array([r['accuracy'] for r in m['per_class']])
    plt.figure(figsize=(7, 3.5))
    plt.hist(acc, bins=20, color='#4f7cff')
    plt.xlabel('per-class accuracy (recall)'); plt.ylabel('# classes')
    plt.title(f"{meta['name']}: mean {acc.mean():.3f}, sd {acc.std():.3f}")
    plt.show()
    print('가장 어려운 클래스:', [r['name'] for r in m['worst_classes'][:5]])

## 4. 두 모델 비교: McNemar 검정

같은 test set에서 두 모델이 서로 다르게 맞힌 이미지 수(b, c)로 정확도 차이가 유의한지 검정합니다.

In [ ]:
same = {}
for meta, m, d in evals:
    same.setdefault(m['dataset'], []).append((meta, m, d))
for ds, items in same.items():
    if len(items) < 2:
        continue
    (ma, ea, da), (mb, eb, db) = items[-2], items[-1]
    za, zb = np.load(da / 'preds.npz'), np.load(db / 'preds.npz')
    r = mcnemar(za['y_true'] == za['y_pred'], zb['y_true'] == zb['y_pred'])
    print(f"[{ds}] A={ma['name']} ({ea['top1']:.4f})  B={mb['name']} ({eb['top1']:.4f})")
    print(f"  A만 맞힘 b={r['only_a_correct']}, B만 맞힘 c={r['only_b_correct']}, p={r['p_value']:.4g} ({r['method']})")